<a href="https://colab.research.google.com/github/srijabiswas-01/audio_analysis/blob/main/audio_spliter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Audio Separation & Analysis Pipeline

## Features

- Upload audio in any common format
- Validate maximum duration (8 minutes)
- Convert audio to WAV
- Separate into:
  - Vocals
  - Drums
  - Bass
  - Other
- Automatically detect output folder
- Package stems into a ZIP archive

# Install Dependencies


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install demucs librosa soundfile ffmpeg-python plotly pandas mutagen ipywidgets

# Imports

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, shutil, os
import librosa, soundfile as sf
import numpy as np, pandas as pd

# Configuration

In [ ]:
MAX_DURATION_SECONDS = 8 * 60

print("=" * 60)
print("Audio Separation Pipeline")
print("=" * 60)

# Upload Audio

In [ ]:
uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file uploaded.")

input_filename = next(iter(uploaded.keys()))

print("Uploaded File:", input_filename)

# Duration Validation

In [ ]:
try:
    duration = librosa.get_duration(path=input_filename)
except TypeError:
    y, sr = librosa.load(
        input_filename,
        sr=None,
        mono=True
    )
    duration = librosa.get_duration(
        y=y,
        sr=sr
    )

minutes = int(duration // 60)
seconds = int(duration % 60)

print(f"Duration : {minutes} min {seconds} sec")

if duration > MAX_DURATION_SECONDS:
    raise ValueError(
        f"""
Audio exceeds maximum duration.

Maximum : 8 minutes
Current : {minutes} min {seconds} sec
"""
    )

print("Duration check passed.")

# Convert to Standard WAV

In [ ]:
converted_path = "input_audio.wav"

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        input_filename,
        "-ar",
        "44100",
        "-ac",
        "2",
        converted_path,
    ],
    check=True,
)

print("Converted to:", converted_path)

# Run Demucs

In [ ]:
subprocess.run(
    [
        "python",
        "-m",
        "demucs",
        "--mp3",
        "--filename",
        "{track}/{stem}.{ext}",
        converted_path,
    ],
    check=True,
)

print("Demucs separation completed.")

# Detect Output Folder Automatically

In [ ]:
demucs_root = Path("separated/htdemucs")

track_dirs = [
    p
    for p in demucs_root.iterdir()
    if p.is_dir()
]

if len(track_dirs) == 0:
    raise RuntimeError(
        "No Demucs output directory found."
    )

root = track_dirs[0]

print("Detected output folder:")
print(root)

# List Generated Files

In [ ]:
print()

print("Generated Files")

for file in root.glob("*"):
    print(file.name)

In [ ]:
# root = Path("separated/htdemucs/input_audio")
# print("Generated files:")
# for f in root.glob("*"):
#     print("-", f)

# Create ZIP Archive

In [ ]:
zip_path = shutil.make_archive(
    "demucs_output",
    "zip",
    root_dir=root
)

print()
print("ZIP created:")
print(zip_path)

# Automatic Download

In [ ]:
# files.download(zip_path)

In [ ]:
!pip -q install plotly mutagen pandas

# Stem Paths

In [ ]:
stem_files = {
    "Vocals": root / "vocals.mp3",
    "Drums": root / "drums.mp3",
    "Bass": root / "bass.mp3",
    "Other": root / "other.mp3",
}

print()
print("Stem dictionary ready.")

# 📊 Audio Analysis & Visualization

## Features

- Stem metadata
- Interactive audio playback
- Waveform plots
- Spectrograms
- FFT spectrum
- BPM estimation
- Export metadata to CSV

# AUDIO ANALYSIS & VISUALIZATION

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import librosa.display

from IPython.display import Audio, display

# Metadata Extraction

In [ ]:
metadata_rows = []

print("=" * 70)
print("STEM METADATA")
print("=" * 70)

for stem_name, file_path in stem_files.items():

    if not file_path.exists():
        print(f"Skipping missing file: {file_path}")
        continue

    y, sr = librosa.load(file_path, sr=None, mono=True)

    duration = librosa.get_duration(y=y, sr=sr)

    rms = float(np.sqrt(np.mean(y ** 2)))

    peak = float(np.max(np.abs(y)))

    filesize = file_path.stat().st_size / (1024 * 1024)

    # Estimate BPM
    try:
        tempo, beats = librosa.beat.beat_track(
            y=y,
            sr=sr
        )

        tempo = float(np.asarray(tempo).squeeze())

    except Exception:

        tempo = np.nan

    metadata_rows.append(
        {
            "Stem": stem_name,
            "File": file_path.name,
            "Duration (sec)": round(duration, 2),
            "Sample Rate": sr,
            "RMS": round(rms, 5),
            "Peak": round(peak, 5),
            "Estimated BPM": round(tempo, 2)
            if not np.isnan(tempo)
            else None,
            "File Size (MB)": round(filesize, 2),
        }
    )

metadata_df = pd.DataFrame(metadata_rows)

display(metadata_df)

# Save Metadata

In [ ]:
metadata_df.to_csv(
    "audio_metadata.csv",
    index=False
)

print("Saved metadata: audio_metadata.csv")

# Interactive Audio Preview

In [ ]:
print()
print("=" * 70)
print("AUDIO PLAYERS")
print("=" * 70)

for stem_name, file_path in stem_files.items():

    if file_path.exists():

        print(stem_name)

        display(Audio(str(file_path)))

# Interactive Waveforms

In [ ]:
# print()
# print("=" * 70)
# print("WAVEFORMS")
# print("=" * 70)

# for stem_name, file_path in stem_files.items():

#     if not file_path.exists():
#         continue

#     y, sr = librosa.load(
#         file_path,
#         sr=None,
#         mono=True
#     )

#     time_axis = np.linspace(
#         0,
#         len(y) / sr,
#         len(y)
#     )

#     fig = go.Figure()

#     fig.add_trace(

#         go.Scatter(

#             x=time_axis,

#             y=y,

#             mode="lines",

#             name=stem_name

#         )

#     )

#     fig.update_layout(

#         title=f"{stem_name} Waveform",

#         xaxis_title="Time (seconds)",

#         yaxis_title="Amplitude",

#         height=350,

#     )

#     fig.show()

# Spectrograms

In [ ]:
# print()
# print("=" * 70)
# print("SPECTROGRAMS")
# print("=" * 70)

# for stem_name, file_path in stem_files.items():

#     if not file_path.exists():
#         continue

#     y, sr = librosa.load(
#         file_path,
#         sr=None,
#         mono=True
#     )

#     plt.figure(figsize=(14, 4))

#     D = librosa.amplitude_to_db(

#         np.abs(

#             librosa.stft(y)

#         ),

#         ref=np.max,

#     )

#     librosa.display.specshow(

#         D,

#         sr=sr,

#         x_axis="time",

#         y_axis="log",

#     )

#     plt.title(

#         f"{stem_name} Spectrogram"

#     )

#     plt.colorbar()

#     plt.show()

# Frequency Spectrum (FFT)

In [ ]:
# print()
# print("=" * 70)
# print("FFT SPECTRUM")
# print("=" * 70)

# for stem_name, file_path in stem_files.items():

#     if not file_path.exists():
#         continue

#     y, sr = librosa.load(
#         file_path,
#         sr=None,
#         mono=True
#     )

#     fft = np.abs(

#         np.fft.rfft(y)

#     )

#     freqs = np.fft.rfftfreq(

#         len(y),

#         1 / sr

#     )

#     fig = go.Figure()

#     fig.add_trace(

#         go.Scatter(

#             x=freqs,

#             y=fft,

#             mode="lines",

#             name=stem_name,

#         )

#     )

#     fig.update_layout(

#         title=f"{stem_name} Frequency Spectrum",

#         xaxis_title="Frequency (Hz)",

#         yaxis_title="Magnitude",

#         height=350,

#     )

#     fig.show()

# Summary

In [ ]:
print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(metadata_df)

# 🎼 Vocal Pitch Shifting & Full Song Reconstruction

## Features

- Visualize vocal pitch (F0)
- Shift vocals by any number of semitones
- Keep instrumental stems unchanged
- Remix:
  - Shifted Vocals
  - Drums
  - Bass
  - Other
- Normalize output
- Save reconstructed song

# PITCH ANALYSIS & REMIXING

In [ ]:
import librosa
import librosa.display
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import Audio, display

# Pitch Visualization (F0)

In [ ]:
print("=" * 70)
print("PITCH (F0) ANALYSIS")
print("=" * 70)

for stem_name, file_path in stem_files.items():

    if not file_path.exists():
        continue

    try:

        y, sr = librosa.load(
            file_path,
            sr=None,
            mono=True
        )

        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7")
        )

        t = librosa.times_like(
            f0,
            sr=sr
        )

        plt.figure(figsize=(14,4))

        plt.plot(
            t,
            f0,
            linewidth=1
        )

        plt.xlabel("Time (seconds)")
        plt.ylabel("Frequency (Hz)")

        plt.title(
            f"{stem_name} Pitch Contour"
        )

        plt.grid(True)

        plt.show()

    except Exception as e:

        print(
            f"Pitch estimation skipped for {stem_name}"
        )

        print(e)

# Pitch Shift Configuration

In [ ]:
SHIFT_SEMITONES = 3

print()
print(
    f"Pitch shifting vocals by {SHIFT_SEMITONES} semitones..."
)

# Load Vocals

In [ ]:
vocals_path = stem_files["Vocals"]

vocals, sr = librosa.load(
    vocals_path,
    sr=None,
    mono=False
)

# Apply Pitch Shift

In [ ]:
shifted_vocals = librosa.effects.pitch_shift(

    vocals,

    sr=sr,

    n_steps=SHIFT_SEMITONES

)

shifted_vocal_file = (
    f"vocals_pitch_{SHIFT_SEMITONES:+d}.wav"
)

sf.write(

    shifted_vocal_file,

    shifted_vocals.T
    if shifted_vocals.ndim > 1
    else shifted_vocals,

    sr

)

print("Saved:", shifted_vocal_file)

display(Audio(shifted_vocal_file))

# Load Instrumental Stems

In [ ]:
drums, _ = librosa.load(
    stem_files["Drums"],
    sr=sr,
    mono=False
)

bass, _ = librosa.load(
    stem_files["Bass"],
    sr=sr,
    mono=False
)

other, _ = librosa.load(
    stem_files["Other"],
    sr=sr,
    mono=False
)


# Trim All Signals

In [ ]:
minimum_length = min(

    shifted_vocals.shape[-1],

    drums.shape[-1],

    bass.shape[-1],

    other.shape[-1],

)

shifted_vocals = shifted_vocals[..., :minimum_length]

drums = drums[..., :minimum_length]

bass = bass[..., :minimum_length]

other = other[..., :minimum_length]


# Gain Controls

In [ ]:
VOCAL_GAIN = 1.0

DRUM_GAIN = 1.0

BASS_GAIN = 1.0

OTHER_GAIN = 1.0

# Remix

In [ ]:
mix = (

    VOCAL_GAIN * shifted_vocals +

    DRUM_GAIN * drums +

    BASS_GAIN * bass +

    OTHER_GAIN * other

)

# Normalize

In [ ]:
peak = np.max(

    np.abs(mix)

)

if peak > 1.0:

    mix = mix / peak

# Save Outputs

In [ ]:
output_mix = "full_song_pitch_changed.wav"

sf.write(

    output_mix,

    mix.T if mix.ndim > 1 else mix,

    sr

)

print()

print("Saved reconstructed song:")

print(output_mix)

# Audio Preview

In [ ]:
display(Audio(output_mix))

# Compare Original Vocal vs Shifted Vocal

In [ ]:
print()

print("=" * 70)
print("ORIGINAL VOCALS")
print("=" * 70)

display(Audio(str(vocals_path)))

print("=" * 70)
print("SHIFTED VOCALS")
print("=" * 70)

display(Audio(shifted_vocal_file))

print("=" * 70)
print("FINAL REMIX")
print("=" * 70)

display(Audio(output_mix))

# INTERACTIVE REMIXER & EXPORT

In [ ]:
import os
import zipfile
import librosa
import soundfile as sf
import numpy as np
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, Audio, clear_output

# VERIFY STEMS

In [ ]:
required = [
    stem_files["Vocals"],
    stem_files["Drums"],
    stem_files["Bass"],
    stem_files["Other"],
]

for f in required:
    if not Path(f).exists():
        raise FileNotFoundError(f"Missing file: {f}")

print("All stems located successfully.")

# NTERACTIVE CONTROLS

In [ ]:
pitch_slider = widgets.IntSlider(
    value=0,
    min=-12,
    max=12,
    step=1,
    description="Pitch"
)

vocal_slider = widgets.FloatSlider(
    value=1.0,
    min=0,
    max=2,
    step=0.1,
    description="Vocals"
)

drum_slider = widgets.FloatSlider(
    value=1.0,
    min=0,
    max=2,
    step=0.1,
    description="Drums"
)

bass_slider = widgets.FloatSlider(
    value=1.0,
    min=0,
    max=2,
    step=0.1,
    description="Bass"
)

other_slider = widgets.FloatSlider(
    value=1.0,
    min=0,
    max=2,
    step=0.1,
    description="Other"
)

apply_button = widgets.Button(
    description="Generate Remix",
    button_style="success"
)

output_box = widgets.Output()

# REMIX FUNCTION

In [ ]:
def generate_remix(_):

    with output_box:

        clear_output()

        print("=" * 60)
        print("Generating Remix")
        print("=" * 60)

        shift = pitch_slider.value

        vocals, sr = librosa.load(
            stem_files["Vocals"],
            sr=None,
            mono=False
        )

        drums, _ = librosa.load(
            stem_files["Drums"],
            sr=sr,
            mono=False
        )

        bass, _ = librosa.load(
            stem_files["Bass"],
            sr=sr,
            mono=False
        )

        other, _ = librosa.load(
            stem_files["Other"],
            sr=sr,
            mono=False
        )

        shifted = librosa.effects.pitch_shift(
            vocals,
            sr=sr,
            n_steps=shift
        )

        L = min(
            shifted.shape[-1],
            drums.shape[-1],
            bass.shape[-1],
            other.shape[-1],
        )

        shifted = shifted[..., :L]
        drums = drums[..., :L]
        bass = bass[..., :L]
        other = other[..., :L]

        remix = (

            vocal_slider.value * shifted +

            drum_slider.value * drums +

            bass_slider.value * bass +

            other_slider.value * other

        )

        peak = np.max(np.abs(remix))

        if peak > 1:
            remix = remix / peak

        outfile = "interactive_remix.wav"

        sf.write(
            outfile,
            remix.T if remix.ndim > 1 else remix,
            sr
        )

        print("Pitch Shift :", shift)
        print("Vocals Gain :", vocal_slider.value)
        print("Drums Gain  :", drum_slider.value)
        print("Bass Gain   :", bass_slider.value)
        print("Other Gain  :", other_slider.value)

        print()
        print("Saved:", outfile)

        display(Audio(outfile))

apply_button.on_click(generate_remix)

display(

    widgets.VBox(

        [

            pitch_slider,

            vocal_slider,

            drum_slider,

            bass_slider,

            other_slider,

            apply_button,

            output_box,

        ]

    )

)

# SAVE METADATA REPORT

In [ ]:
print()
print("=" * 60)
print("Saving metadata report...")
print("=" * 60)

metadata_df.to_csv(
    "audio_metadata_report.csv",
    index=False
)

print("Saved audio_metadata_report.csv")

# CREATE ZIP OF RESULTS

In [ ]:
print()
print("=" * 60)
print("Creating ZIP package...")
print("=" * 60)

files_to_include = []

for ext in [

    "*.wav",

    "*.mp3",

    "*.csv",

]:

    files_to_include.extend(

        Path(".").glob(ext)

    )

zip_name = "audio_analysis_results.zip"

with zipfile.ZipFile(

    zip_name,

    "w",

    zipfile.ZIP_DEFLATED

) as z:

    for file in files_to_include:

        z.write(

            file,

            arcname=file.name

        )

print("Created:", zip_name)

In [ ]:
# files.download(zip_name)